In [ ]:
import os
import pandas as pd
import nibabel as nib
import numpy as np
import cv2

# --- Paths ---
csv_file = r"C:\Users\ASUS\Downloads\ADNI_labels.csv"  # <-- your CSV file
adni_root = r"C:\Users\ASUS\Downloads\ADNI1_Complete 1Yr 1.5T1\ADNI"
output_root = r"C:\Users\ASUS\Downloads\ADNI_preprocessed"

# --- Load labels ---
df = pd.read_csv(csv_file)

#  Check the first few rows to confirm column names
print(df.head())

# Adjust column names  Subject column = "Subject ID", group column = "Group"
labels = dict(zip(df["Subject"], df["Group"]))  # {"133_S_0488": "MCI", "002_S_0295": "CN", ...}

# --- Traverse dataset ---
for subject_id, group in labels.items():
    subject_folder = os.path.join(adni_root, subject_id)
    if not os.path.exists(subject_folder):
        print(f"❌ Skipped {subject_id} (folder not found)")
        continue

    # Walk through this subject’s folder to find .nii files
    for root, _, files in os.walk(subject_folder):
        for file in files:
            if file.endswith(".nii"):
                nii_path = os.path.join(root, file)

                try:
                    # Load NIfTI
                    img = nib.load(nii_path)
                    data = img.get_fdata()

                    # Create output folder for this group
                    out_dir = os.path.join(output_root, group)
                    os.makedirs(out_dir, exist_ok=True)

                    # Save slices
                    for i in range(data.shape[2]):
                        slice_2d = data[:, :, i]
                        slice_2d = cv2.normalize(slice_2d, None, 0, 255, cv2.NORM_MINMAX)
                        slice_2d = slice_2d.astype(np.uint8)

                        out_name = f"{subject_id}_slice{i}.jpg"
                        out_path = os.path.join(out_dir, out_name)
                        cv2.imwrite(out_path, slice_2d)

                    print(f"✅ Processed {subject_id} → {group} ({data.shape[2]} slices)")

                except Exception as e:
                    print(f"❌ Error processing {nii_path}: {e}")


  Image Data ID     Subject Group Sex  Age Visit Modality  \
0        I97327  941_S_1311   MCI   M   69    sc      MRI   
1       I112538  941_S_1311   MCI   M   70   m12      MRI   
2        I97341  941_S_1311   MCI   M   70   m06      MRI   
3        I63874  941_S_1202    CN   M   78    sc      MRI   
4        I75150  941_S_1202    CN   M   78   m06      MRI   

                                  Description       Type   Acq Date Format  \
0    MPR; GradWarp; B1 Correction; N3; Scaled  Processed  3/02/2007  NiFTI   
1    MPR; GradWarp; B1 Correction; N3; Scaled  Processed  6/01/2008  NiFTI   
2  MPR-R; GradWarp; B1 Correction; N3; Scaled  Processed  9/27/2007  NiFTI   
3  MPR-R; GradWarp; B1 Correction; N3; Scaled  Processed  1/30/2007  NiFTI   
4    MPR; GradWarp; B1 Correction; N3; Scaled  Processed  8/24/2007  NiFTI   

  Downloaded  
0        NaN  
1        NaN  
2        Yes  
3        NaN  
4        NaN  
✅ Processed 941_S_1311 → MCI (160 slices)
❌ Skipped 941_S_1202 (folder not